# CP4 — Task 2: Perbandingan Baseline dan Kontribusi Komponen (W11) — Revisi

**Sumber utama:** `outputs/w11/w11_all_configs.csv` dan `outputs/w11/w11_metadata.json`, hasil **Task 1**. Tidak mengulang eksperimen, tidak memakai data test.

**Output:** (1) tabel 5 model; (2) delta absolut dan relatif terhadap Popularity; (3) kontribusi CF murni α=1, CBF murni α=0, weighted α=0,3 dan RRF terbaik; (4) Hit Rate@10 dan Catalog Coverage@10; (5) interpretasi hasil yang menggunakan angka aktual.

**Perbaikan penting:** metrik Hit Rate dan `unique_items@10` **diambil dari log Task 1 / metadata**. Tidak perlu berkas `w11_recommendations.csv` maupun `w11_validation_relevant.csv`. Semua nilai yang tak tersedia akan ditandai `NaN`, bukan direkayasa. Catalog coverage adalah `unique_items@10 / candidate_universe` (fraksi), disajikan juga dalam persen.

## 1. Setup dan lokasi hasil Task 1

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "outputs" / "w11" / "w11_all_configs.csv").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Tidak menemukan outputs/w11/w11_all_configs.csv. Jalankan Task 1 lebih dulu dan buka Task 2 dari proyek yang sama.")
OUT = PROJECT_ROOT / "outputs" / "w11"
TASK2_OUT = OUT / "task2"
TASK2_OUT.mkdir(parents=True, exist_ok=True)
K = 10
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
print("Root proyek:", PROJECT_ROOT)
print("Hasil Task 1:", OUT)
print("Hasil Task 2:", TASK2_OUT)

C:\Users\Rache\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Root proyek: d:\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems
Hasil Task 1: d:\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems\outputs\w11
Hasil Task 2: d:\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems\outputs\w11\task2


## 2. Muat, audit, dan gabungkan metrik agregat Task 1

In [2]:
log = pd.read_csv(OUT / "w11_all_configs.csv")
meta_path = OUT / "w11_metadata.json"
if not meta_path.is_file():
    raise FileNotFoundError(f"Metadata Task 1 tidak ada: {meta_path}")
metadata = json.loads(meta_path.read_text(encoding="utf-8"))
protocol = metadata.get("protocol", {})
if protocol.get("uses_final_test") is not False:
    raise ValueError("Metadata tidak memastikan test set belum digunakan; hentikan dahulu.")
required = {"config_id", "family", "ndcg@10", "recall@10"}
if missing := required - set(log.columns):
    raise ValueError(f"Kolom wajib Task 1 hilang: {sorted(missing)}")
if log["config_id"].duplicated().any():
    raise ValueError("config_id pada Task 1 tidak unik.")

# Metadata 'winners' menyimpan metrik agregat sebagai cadangan jika beberapa
# kolom tidak diekspor ke w11_all_configs.csv.
winners = metadata.get("winners", {})
win_map = {}
for item in [winners.get("best_overall"), *winners.get("best_by_family", {}).values()]:
    if isinstance(item, dict) and item.get("config_id"):
        win_map[item["config_id"]] = item

aggregation_cols = ["users_evaluated", "users_with_hit@10", "hit_rate_pct@10", "unique_items@10"]
for col in aggregation_cols:
    if col not in log:
        log[col] = np.nan
    log[col] = pd.to_numeric(log[col], errors="coerce")
    # Cadangan hanya untuk konfigurasi winner dengan ID yang cocok; tidak
    # menyebarkan hasil winner ke konfigurasi model lain.
    missing_mask = log[col].isna()
    log.loc[missing_mask, col] = log.loc[missing_mask, "config_id"].map(
        lambda cid: win_map.get(cid, {}).get(col, np.nan)
    )

for col in [c for c in log.columns if "@" in c or c == "duration_sec"]:
    log[col] = pd.to_numeric(log[col], errors="coerce")

# Rekonstruksi hanya jika users_with_hit dan users_evaluated tersedia.
missing_hit = log["hit_rate_pct@10"].isna()
valid_denominator = log["users_evaluated"].gt(0)
mask = missing_hit & valid_denominator & log["users_with_hit@10"].notna()
log.loc[mask, "hit_rate_pct@10"] = (100 * log.loc[mask, "users_with_hit@10"] / log.loc[mask, "users_evaluated"])

UNIVERSE = int(protocol["candidate_universe"])
if UNIVERSE <= 0:
    raise ValueError("candidate_universe harus > 0")
EXPECTED_USERS = int(protocol.get("evaluable_users", 0))
log["catalog_coverage@10"] = log["unique_items@10"] / UNIVERSE
log["catalog_coverage_pct@10"] = log["catalog_coverage@10"] * 100
log["hit_rate@10"] = log["hit_rate_pct@10"] / 100  # fraksi 0–1

if log["unique_items@10"].notna().any() and (log["unique_items@10"].dropna() > UNIVERSE).any():
    raise ValueError("Jumlah item unik melebihi candidate_universe; periksa sumber data.")
if EXPECTED_USERS and log["users_evaluated"].notna().any():
    bad = log[log["users_evaluated"].notna() & (log["users_evaluated"] != EXPECTED_USERS)]
    if not bad.empty:
        raise ValueError("users_evaluated tidak konsisten dengan protocol: " + ", ".join(bad.config_id))

print(f"Konfigurasi: {len(log)} | User dievaluasi: {EXPECTED_USERS:,} | Kandidat: {UNIVERSE:,}")
print("Metrik Top-10 tersedia (jumlah konfigurasi):")
print(log[["hit_rate_pct@10", "unique_items@10", "catalog_coverage_pct@10"]].notna().sum())
display(log[["config_id", "family", "ndcg@10", "recall@10", "hit_rate_pct@10", "unique_items@10"]]
        .sort_values(["ndcg@10", "recall@10"], ascending=False).head(10).round(6))

Konfigurasi: 20 | User dievaluasi: 11,659 | Kandidat: 22,753
Metrik Top-10 tersedia (jumlah konfigurasi):
hit_rate_pct@10            20
unique_items@10            20
catalog_coverage_pct@10    20
dtype: int64


C:\Users\Rache\AppData\Local\Temp\ipykernel_12072\3454287344.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  log.loc[missing_mask, col] = log.loc[missing_mask, "config_id"].map(
C:\Users\Rache\AppData\Local\Temp\ipykernel_12072\3454287344.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  log.loc[missing_mask, col] = log.loc[missing_mask, "config_id"].map(
C:\Users\Rache\AppData\Local\Temp\ipykernel_12072\3454287344.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype

,config_id,family,ndcg@10,recall@10,hit_rate_pct@10,unique_items@10
0,E1-popularity,baseline,0.009222,0.013499,3.104898,18
19,E5-rrf-k100,rrf,0.009086,0.014198,2.753238,18897
17,E5-rrf-k60,rrf,0.008750,0.013565,2.453041,19122
18,E5-rrf-k80,rrf,0.008661,0.013259,2.444464,19121
16,E5-rrf-k40,rrf,0.008291,0.012633,2.30723,19174
6,E4-weighted-a030,weighted,0.007930,0.012820,2.187152,17725
15,E5-rrf-k20,rrf,0.007841,0.012407,2.212883,19100
5,E4-weighted-a020,weighted,0.007774,0.012455,2.16142,17106
7,E4-weighted-a040,weighted,0.007726,0.012401,2.092804,18439
4,E4-weighted-a010,weighted,0.007703,0.012670,2.169997,16915


## 3. Lima model baseline vs track

RRF dipilih berdasarkan **NDCG@10** tertinggi (Recall@10 sebagai *tie-breaker*) mengikuti Task 1. Jika metadata menyebut pemenang RRF, notebook memverifikasi bahwa konfigurasi tersebut ada di log.

In [3]:
BASELINES = ["E1-popularity", "E2-itemcf-k100", "E3-tfidf-cbf"]
WEIGHTED = "E4-weighted-a030"
rrf_candidates = log[log["family"].astype(str).str.casefold().eq("rrf")]
if rrf_candidates.empty:
    raise ValueError("Tidak ditemukan konfigurasi RRF.")
rrf_best = rrf_candidates.sort_values(["ndcg@10", "recall@10"], ascending=False).iloc[0]["config_id"]
selected_ids = BASELINES + [WEIGHTED, rrf_best]
if missing := set(selected_ids) - set(log["config_id"]):
    raise ValueError(f"Konfigurasi berikut tidak ada pada log: {sorted(missing)}")
labels = dict(zip(selected_ids, ["Popularity", "Item-CF", "TF-IDF CBF", "Weighted α=0,3", "RRF terbaik"]))
selected = log.set_index("config_id").loc[selected_ids].reset_index()
selected.insert(0, "model", selected["config_id"].map(labels))
metric_cols = [f"{m}@{k}" for k in [10, 20, 50]
               for m in ["recall", "precision", "ndcg", "mrr", "map"]
               if f"{m}@{k}" in selected.columns]
comparison = selected[["model", "config_id", "family", *metric_cols,
                       "hit_rate_pct@10", "unique_items@10", "catalog_coverage_pct@10"]].copy()
display(comparison[["model", "config_id", "ndcg@10", "recall@10", "precision@10",
                    "mrr@10", "map@10", "hit_rate_pct@10", "unique_items@10", "catalog_coverage_pct@10"]].round(6))
comparison.to_csv(TASK2_OUT / "task2_lima_model.csv", index=False)
print("RRF terpilih:", rrf_best)

,model,config_id,ndcg@10,recall@10,precision@10,mrr@10,map@10,hit_rate_pct@10,unique_items@10,catalog_coverage_pct@10
0,Popularity,E1-popularity,0.009222,0.013499,0.003182,0.012988,0.005490,3.104898,18,0.079110
1,Item-CF,E2-itemcf-k100,0.003323,0.005800,0.001141,0.003536,0.001981,1.089287,18129,79.677405
2,TF-IDF CBF,E3-tfidf-cbf,0.007491,0.012148,0.002196,0.008095,0.004955,2.118535,16851,74.060563
3,"Weighted α=0,3",E4-weighted-a030,0.007930,0.012820,0.002281,0.008818,0.005222,2.187152,17725,77.901815
4,RRF terbaik,E5-rrf-k100,0.009086,0.014198,0.002873,0.010687,0.005873,2.753238,18897,83.052784


RRF terpilih: E5-rrf-k100


## 4. Delta absolut dan relatif terhadap Popularity

Untuk setiap metrik, `Δ absolut = nilai model − nilai Popularity` dan `Δ relatif (%) = Δ absolut / nilai Popularity × 100`. Untuk metrik dalam fraksi (NDCG, Recall, dll.), delta absolut mempertahankan skala 0–1. Untuk Hit Rate dan coverage di tabel ini, delta absolut dihitung dalam **poin persentase** karena nilai yang dibandingkan sudah dalam persen. Nilai relatif tidak didefinisikan jika nilai baseline adalah nol atau tidak tersedia.

In [4]:
pop = selected.set_index("config_id").loc["E1-popularity"]
delta_metrics = metric_cols + ["hit_rate_pct@10", "catalog_coverage_pct@10"]
delta = selected[["model", "config_id"]].copy()
for m in delta_metrics:
    delta[f"{m} | absolut"] = selected[m].to_numpy(dtype=float) - float(pop[m])
    base_value = float(pop[m])
    delta[f"{m} | relatif (%)"] = (100 * delta[f"{m} | absolut"] / base_value
                                      if np.isfinite(base_value) and base_value != 0 else np.nan)
focus = ["ndcg@10", "recall@10", "precision@10", "mrr@10", "map@10",
         "hit_rate_pct@10", "catalog_coverage_pct@10"]
display(delta[["model"] + [col for m in focus
                           for col in [f"{m} | absolut", f"{m} | relatif (%)"]]].round(6))
delta.to_csv(TASK2_OUT / "task2_delta_vs_popularity.csv", index=False)

,model,ndcg@10 | absolut,ndcg@10 | relatif (%),recall@10 | absolut,recall@10 | relatif (%),precision@10 | absolut,precision@10 | relatif (%),mrr@10 | absolut,mrr@10 | relatif (%),map@10 | absolut,map@10 | relatif (%),hit_rate_pct@10 | absolut,hit_rate_pct@10 | relatif (%),catalog_coverage_pct@10 | absolut,catalog_coverage_pct@10 | relatif (%)
0,Popularity,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,Item-CF,-0.005899,-63.966419,-0.007699,-57.035884,-0.002041,-64.150943,-0.009452,-72.774563,-0.003509,-63.919639,-2.015610,-64.917127,79.598295,100616.666667
2,TF-IDF CBF,-0.001731,-18.772116,-0.001350,-10.003316,-0.000986,-30.997305,-0.004892,-37.668899,-0.000535,-9.745575,-0.986362,-31.767956,73.981453,93516.666667
3,"Weighted α=0,3",-0.001292,-14.007495,-0.000678,-5.023007,-0.000901,-28.301887,-0.004170,-32.104259,-0.000267,-4.871427,-0.917746,-29.558011,77.822705,98372.222222
4,RRF terbaik,-0.000136,-1.469854,0.000700,5.185520,-0.000309,-9.703504,-0.002300,-17.711773,0.000383,6.976966,-0.351660,-11.325967,82.973674,104883.333333


## 5. Kontribusi komponen (α=1, α=0, α=0,3, RRF)

In [5]:
contrib_ids = ["E4-weighted-a100", "E4-weighted-a000", "E4-weighted-a030", rrf_best]
contrib_names = ["CF murni (α=1)", "CBF murni (α=0)", "Weighted α=0,3", "RRF terbaik"]
if missing := set(contrib_ids) - set(log["config_id"]):
    raise ValueError(f"Konfigurasi kontribusi komponen tidak ditemukan: {sorted(missing)}")
contribution = log.set_index("config_id").loc[contrib_ids].reset_index()
contribution.insert(0, "komponen", contrib_names)
keep = ["komponen", "config_id", "ndcg@10", "recall@10", "precision@10", "mrr@10",
        "map@10", "hit_rate_pct@10", "unique_items@10", "catalog_coverage_pct@10"]
contribution = contribution[[c for c in keep if c in contribution]]
display(contribution.round(6))
contribution.to_csv(TASK2_OUT / "task2_kontribusi_komponen.csv", index=False)

ablation_rows = []
base = contribution.set_index("komponen")
for fusion in contrib_names[2:]:
    for component in contrib_names[:2]:
        ablation_rows.append({
            "fusi": fusion, "dibanding_komponen": component,
            "delta_ndcg@10": base.loc[fusion, "ndcg@10"] - base.loc[component, "ndcg@10"],
            "delta_recall@10": base.loc[fusion, "recall@10"] - base.loc[component, "recall@10"],
        })
ablation = pd.DataFrame(ablation_rows)
display(ablation.round(6))
ablation.to_csv(TASK2_OUT / "task2_delta_kontribusi.csv", index=False)

,komponen,config_id,ndcg@10,recall@10,precision@10,mrr@10,map@10,hit_rate_pct@10,unique_items@10,catalog_coverage_pct@10
0,CF murni (α=1),E4-weighted-a100,0.003328,0.005757,0.001132,0.003562,0.002000,1.08071,18153,79.782886
1,CBF murni (α=0),E4-weighted-a000,0.007502,0.012148,0.002196,0.008138,0.004963,2.118535,16851,74.060563
2,"Weighted α=0,3",E4-weighted-a030,0.007930,0.012820,0.002281,0.008818,0.005222,2.187152,17725,77.901815
3,RRF terbaik,E5-rrf-k100,0.009086,0.014198,0.002873,0.010687,0.005873,2.753238,18897,83.052784


,fusi,dibanding_komponen,delta_ndcg@10,delta_recall@10
0,"Weighted α=0,3",CF murni (α=1),0.004602,0.007064
1,"Weighted α=0,3",CBF murni (α=0),0.000429,0.000672
2,RRF terbaik,CF murni (α=1),0.005758,0.008442
3,RRF terbaik,CBF murni (α=0),0.001585,0.002050


## 6. Hit Rate dan Catalog Coverage@10 dari agregat Task 1

- **Hit Rate@10:** proporsi user yang memiliki setidaknya satu item relevan di 10 rekomendasi pertama. `hit_rate_pct@10` dari Task 1 dipertahankan dalam satuan **persen**.
- **Coverage@10:** `unique_items@10 / candidate_universe`, menggunakan **22.753 kandidat** sesuai metadata eksperimen (angka dibaca dari metadata saat dijalankan).
- Metrik ini tersedia tanpa merekonstruksi ranking per-user. Apabila Task 1 belum mengekspor kolom untuk model tertentu, tampil `NaN` dan catatan keterbatasan. Tidak menggunakan `interactions_test`.


In [6]:
hr_coverage = selected[["model", "config_id", "users_evaluated", "users_with_hit@10",
                        "hit_rate@10", "hit_rate_pct@10", "unique_items@10",
                        "catalog_coverage@10", "catalog_coverage_pct@10"]].copy()
hr_coverage["status"] = np.where(
    hr_coverage[["hit_rate_pct@10", "catalog_coverage_pct@10"]].notna().all(axis=1),
    "Lengkap (agregat Task 1)", "Sebagian kolom agregat Task 1 belum tersedia")
display(hr_coverage.round(6))
hr_coverage.to_csv(TASK2_OUT / "task2_hit_rate_coverage.csv", index=False)
if hr_coverage["status"].ne("Lengkap (agregat Task 1)").any():
    print("PERHATIAN: Lengkapi kolom users_with_hit@10 / hit_rate_pct@10 / unique_items@10")
    print("di w11_all_configs.csv untuk konfigurasi terkait, atau ekspor metrik dari Task 1.")

,model,config_id,users_evaluated,users_with_hit@10,hit_rate@10,hit_rate_pct@10,unique_items@10,catalog_coverage@10,catalog_coverage_pct@10,status
0,Popularity,E1-popularity,11659,362,0.031049,3.104898,18,0.000791,0.079110,Lengkap (agregat Task 1)
1,Item-CF,E2-itemcf-k100,11659,127,0.010893,1.089287,18129,0.796774,79.677405,Lengkap (agregat Task 1)
2,TF-IDF CBF,E3-tfidf-cbf,11659,247,0.021185,2.118535,16851,0.740606,74.060563,Lengkap (agregat Task 1)
3,"Weighted α=0,3",E4-weighted-a030,11659,255,0.021872,2.187152,17725,0.779018,77.901815,Lengkap (agregat Task 1)
4,RRF terbaik,E5-rrf-k100,11659,321,0.027532,2.753238,18897,0.830528,83.052784,Lengkap (agregat Task 1)


## 7. Tabel gabungan lima model

In [7]:
final_cols = ["model", "config_id", "ndcg@10", "recall@10", "precision@10", "mrr@10", "map@10",
              "hit_rate_pct@10", "unique_items@10", "catalog_coverage_pct@10"]
final = selected[[c for c in final_cols if c in selected]].copy()
display(final.round(6))
final.to_csv(TASK2_OUT / "task2_ringkasan_lima_model.csv", index=False)

pop_r = selected.set_index("config_id").loc["E1-popularity"]
rrf_r = selected.set_index("config_id").loc[rrf_best]
ndcg_delta = float(rrf_r["ndcg@10"] - pop_r["ndcg@10"])
rec_delta = float(rrf_r["recall@10"] - pop_r["recall@10"])
rel_ndcg = ndcg_delta / float(pop_r["ndcg@10"]) * 100 if pop_r["ndcg@10"] else np.nan
rel_recall = rec_delta / float(pop_r["recall@10"]) * 100 if pop_r["recall@10"] else np.nan
print(f"RRF vs Popularity — Recall@10: {rec_delta:+.6f} ({rel_recall:+.2f}% relatif)")
print(f"RRF vs Popularity — NDCG@10 : {ndcg_delta:+.6f} ({rel_ndcg:+.2f}% relatif)")

,model,config_id,ndcg@10,recall@10,precision@10,mrr@10,map@10,hit_rate_pct@10,unique_items@10,catalog_coverage_pct@10
0,Popularity,E1-popularity,0.009222,0.013499,0.003182,0.012988,0.005490,3.104898,18,0.079110
1,Item-CF,E2-itemcf-k100,0.003323,0.005800,0.001141,0.003536,0.001981,1.089287,18129,79.677405
2,TF-IDF CBF,E3-tfidf-cbf,0.007491,0.012148,0.002196,0.008095,0.004955,2.118535,16851,74.060563
3,"Weighted α=0,3",E4-weighted-a030,0.007930,0.012820,0.002281,0.008818,0.005222,2.187152,17725,77.901815
4,RRF terbaik,E5-rrf-k100,0.009086,0.014198,0.002873,0.010687,0.005873,2.753238,18897,83.052784


RRF vs Popularity — Recall@10: +0.000700 (+5.19% relatif)
RRF vs Popularity — NDCG@10 : -0.000136 (-1.47% relatif)


## 8. Pembahasan — Mengapa RRF unggul Recall tetapi kalah NDCG?

Recall@10 mengukur proporsi target relevan yang ditemukan di 10 rekomendasi, sedangkan NDCG@10 memperhitungkan peringkatnya (relevan pada posisi atas mendapat bobot lebih besar). RRF menggabungkan urutan dari beberapa sumber rekomendasi menggunakan posisi, bukan sekadar menjumlah skor mentah. Karena itu RRF dapat menemukan lebih banyak target relevan secara total, tetapi sebagian berada pada posisi yang kurang menguntungkan untuk NDCG. Ini penjelasan mekanisme yang konsisten dengan pola metrik, bukan bukti langsung tentang lokasi setiap item karena ranking per-user tidak dianalisis di Task 2.

Recall rata-rata yang lebih tinggi tidak otomatis berarti Hit Rate lebih tinggi. Keduanya mengukur hal berbeda. Coverage juga bukan metrik akurasi, melainkan keragaman item yang muncul pada semua daftar Top-10.

In [8]:
hit_pop = pop_r.get("hit_rate_pct@10", np.nan)
hit_rrf = rrf_r.get("hit_rate_pct@10", np.nan)
cov_pop = pop_r.get("catalog_coverage_pct@10", np.nan)
cov_rrf = rrf_r.get("catalog_coverage_pct@10", np.nan)
if rec_delta > 0 and ndcg_delta < 0:
    interpretation = (
        f"**Hasil validation:** RRF `{rrf_best}` memperoleh Recall@10 {rrf_r['recall@10']:.6f} "
        f"vs Popularity {pop_r['recall@10']:.6f} (**{rel_recall:+.2f}%** relatif); "
        f"NDCG@10 RRF {rrf_r['ndcg@10']:.6f} vs Popularity {pop_r['ndcg@10']:.6f} "
        f"(**{rel_ndcg:+.2f}%** relatif). Ini menunjukkan trade-off antara proporsi target ditemukan dan "
        "kualitas posisi ranking menurut NDCG pada data ini. "
    )
else:
    interpretation = "Data yang dimuat tidak menunjukkan pola RRF Recall lebih tinggi sekaligus NDCG lebih rendah daripada Popularity. "
if pd.notna(hit_pop) and pd.notna(hit_rrf):
    interpretation += (f"Hit Rate@10: Popularity **{hit_pop:.4f}%** dan RRF **{hit_rrf:.4f}%**. ")
if pd.notna(cov_pop) and pd.notna(cov_rrf):
    interpretation += (f"Catalog Coverage@10: Popularity **{cov_pop:.4f}%** dan RRF **{cov_rrf:.4f}%**. ")
interpretation += "Angka ini berasal dari **validation Task 1**, bukan test final."
display(Markdown(interpretation))

**Hasil validation:** RRF `E5-rrf-k100` memperoleh Recall@10 0.014198 vs Popularity 0.013499 (**+5.19%** relatif); NDCG@10 RRF 0.009086 vs Popularity 0.009222 (**-1.47%** relatif). Ini menunjukkan trade-off antara proporsi target ditemukan dan kualitas posisi ranking menurut NDCG pada data ini. Hit Rate@10: Popularity **3.1049%** dan RRF **2.7532%**. Catalog Coverage@10: Popularity **0.0791%** dan RRF **83.0528%**. Angka ini berasal dari **validation Task 1**, bukan test final.

## 9. Audit hasil dan simpan ringkasan

In [9]:
report = {
    "task": "CP4 Task 2 — Perbandingan Baseline",
    "source_files": ["outputs/w11/w11_all_configs.csv", "outputs/w11/w11_metadata.json"],
    "split": "validation", "uses_final_test": False, "k": K,
    "candidate_universe": UNIVERSE, "evaluable_users": EXPECTED_USERS,
    "selected_configs": selected_ids, "rrf_best": rrf_best,
    "rrf_vs_popularity": {
        "delta_ndcg@10": ndcg_delta, "relative_ndcg_percent": rel_ndcg,
        "delta_recall@10": rec_delta, "relative_recall_percent": rel_recall,
    },
    "hit_rate_and_coverage_complete": bool(
        hr_coverage[["hit_rate_pct@10", "catalog_coverage_pct@10"]].notna().all().all()),
    "missing_aggregate_configs": hr_coverage.loc[
        hr_coverage["status"] != "Lengkap (agregat Task 1)", "config_id"].tolist(),
}
(TASK2_OUT / "task2_summary.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("HASIL TASK 2 (validation):")
for p in sorted(TASK2_OUT.glob("task2_*")):
    print(" -", p.relative_to(PROJECT_ROOT))
print("Kelengkapan Hit Rate dan Coverage:",
      "LENGKAP" if report["hit_rate_and_coverage_complete"] else "BELUM LENGKAP — lihat konfigurasi yang hilang di metadata")

HASIL TASK 2 (validation):
 - outputs\w11\task2\task2_delta_kontribusi.csv
 - outputs\w11\task2\task2_delta_vs_popularity.csv
 - outputs\w11\task2\task2_hit_rate_coverage.csv
 - outputs\w11\task2\task2_kontribusi_komponen.csv
 - outputs\w11\task2\task2_lima_model.csv
 - outputs\w11\task2\task2_ringkasan_lima_model.csv
 - outputs\w11\task2\task2_summary.json
Kelengkapan Hit Rate dan Coverage: LENGKAP
